# Steam's WebAPI data

Explore the various Endpoints from Steam's WebAPI. 

In [10]:
import requests
import pandas as pd
from datetime import datetime as dt
from src.utils.time_utils import unix_to_datetime 
import numpy as np
from src.config import ROOT_DIR
pd.set_option("display.max_columns", 100)

In [11]:
# Steam API constants
STEAM_URL = 'http://api.steampowered.com/'
LIVE_LEAGUE_GAMES = 'IDOTA2Match_570/GetLiveLeagueGames/v1'
REAL_TIME_STATS = 'IDOTA2MatchStats_570/GetRealtimeStats/v1' # Requires server_steam_id 
API_KEY = 'F0E5D7D11B592792FE20D84FBB745D97'

# Exploring Get Match Details Endpoint

In [12]:
with open(f'{ROOT_DIR}/data/pro_match_ids/dota2_pro_match_ids_20240901.csv', 'r') as file:
    df = pd.read_csv(file)
    
match_ids = df['match_id']

In [13]:
len(match_ids)

500

In [14]:
import json
from retry import retry

pro_match_records = []
session = requests.Session()
session.params.update({'key': API_KEY})

@retry(tries=3, delay=2)
def fetch_match_details(match_id):
    try:
        url = f'{STEAM_URL}IDOTA2Match_570/GetMatchDetails/v1?match_id={match_id}'
        res = session.get(url)
        match_details = res.json()
        if not match_details:
            print(f"Empty dictionary for match_id {match_id}, retrying...")
            raise ValueError(f"Empty dictionary for match_id {match_id}")
        else:
            return match_details
    except Exception as err:
        print(f"Error for match_id {match_id}: {err}")
        raise  

for match_id in match_ids:
    try:
        match_data = fetch_match_details(match_id)
        pro_match_records.append(match_data)
    except Exception:
        # This block will execute if all retries fail and the error needs to be handled at this level.
        print(f"Failed to fetch data for match_id {match_id} after multiple retries.")

# Takes about 1 hour for 10,000 records to be retrieved. 


Empty dictionary for match_id 7923363324, retrying...
Error for match_id 7923363324: Empty dictionary for match_id 7923363324
Empty dictionary for match_id 7923363324, retrying...
Error for match_id 7923363324: Empty dictionary for match_id 7923363324
Empty dictionary for match_id 7923363324, retrying...
Error for match_id 7923363324: Empty dictionary for match_id 7923363324
Failed to fetch data for match_id 7923363324 after multiple retries.
Empty dictionary for match_id 7923347535, retrying...
Error for match_id 7923347535: Empty dictionary for match_id 7923347535
Empty dictionary for match_id 7923347535, retrying...
Error for match_id 7923347535: Empty dictionary for match_id 7923347535
Empty dictionary for match_id 7923347535, retrying...
Error for match_id 7923347535: Empty dictionary for match_id 7923347535
Failed to fetch data for match_id 7923347535 after multiple retries.
Empty dictionary for match_id 7923329639, retrying...
Error for match_id 7923329639: Empty dictionary for 

KeyboardInterrupt: 

### Exploring the underlying structure of each JSON record

In [244]:
## Exploring the top level of a json file
len(pro_match_records)

12220

In [245]:
empty_dictionaries_count = sum(1 for d in pro_match_records if isinstance(d, dict) and not d)
print("number of empty records extracted" , empty_dictionaries_count)

# Too many empty records, need to implement retries. 

number of empty records extracted 0


In [246]:
pro_match_records[0].keys()

dict_keys(['result'])

In [247]:
pro_match_records[0]['result'].keys()

dict_keys(['players', 'radiant_win', 'duration', 'pre_game_duration', 'start_time', 'match_id', 'match_seq_num', 'tower_status_radiant', 'tower_status_dire', 'barracks_status_radiant', 'barracks_status_dire', 'cluster', 'first_blood_time', 'lobby_type', 'human_players', 'leagueid', 'game_mode', 'flags', 'engine', 'radiant_score', 'dire_score', 'radiant_team_id', 'radiant_name', 'radiant_logo', 'radiant_team_complete', 'dire_team_id', 'dire_name', 'dire_logo', 'dire_team_complete', 'radiant_captain', 'dire_captain', 'picks_bans'])

### Columns to use for Model training
1. 'players' - Contains information on player's account_id, hero_id and which team he is on
2. 'radiant_win'
3. 'match_id'
4. 'start_time'
5.  'radiant_team_id'
6.  'radiant_name'
7.  'dire_team_id'
8.  'dire_name'
9.  'game_duration'

Information about these columns can be found at https://rdrr.io/cran/RDota2/man/get_match_details.html



### Analyzing player column

In [248]:
print("number of players: ", len(pro_match_records[0]['result']['players']))
print("keys in players dictionary: ", pro_match_records[0]['result']['players'][0].keys())

number of players:  10
keys in players dictionary:  dict_keys(['account_id', 'player_slot', 'team_number', 'team_slot', 'hero_id', 'item_0', 'item_1', 'item_2', 'item_3', 'item_4', 'item_5', 'backpack_0', 'backpack_1', 'backpack_2', 'item_neutral', 'kills', 'deaths', 'assists', 'leaver_status', 'last_hits', 'denies', 'gold_per_min', 'xp_per_min', 'level', 'net_worth', 'aghanims_scepter', 'aghanims_shard', 'moonshard', 'hero_damage', 'tower_damage', 'hero_healing', 'gold', 'gold_spent', 'scaled_hero_damage', 'scaled_tower_damage', 'scaled_hero_healing', 'ability_upgrades'])


#### Relevant Columns to keep 

1. 'account_id'
2. 'hero_id'
3. 'player_slot'


##### Reasons for ignoring other Columns

Those are end game stats, and hence unlikely to be available right after the ban/pick phase 


### Data Transformation for pro_match_records

In [258]:
def pro_match_template():
    template = {
        'match_id': np.nan,
        'radiant_team_id': np.nan,
        'radiant_name': np.nan,
        'dire_team_id': np.nan,
        'dire_name': np.nan,
        'duration': np.nan,
        'start_time': np.nan,
        'radiant_win': np.nan
        
    }
    
    for i in list(range(0,5)) + list(range(128,133)):
        template[f"{i}_account_id"] = np.nan
        template[f"{i}_hero_id"] = np.nan
        
    return template


In [259]:
## Data Transformation for pro_match_records
list_pro_matches = []

for row in pro_match_records:
    pro_match_dict = pro_match_template()
    result = row.get('result', {})

    if result is not None:
        # Populating other columns
        pro_match_dict['match_id'] = result.get('match_id', np.nan)
        pro_match_dict['radiant_team_id'] = result.get('radiant_team_id', np.nan)
        pro_match_dict['radiant_name'] = result.get('radiant_name', np.nan)
        pro_match_dict['dire_team_id'] = result.get('dire_team_id', np.nan)
        pro_match_dict['dire_name'] = result.get('dire_name', np.nan)
        pro_match_dict['duration'] = result.get('duration', np.nan)
        pro_match_dict['start_time'] = result.get('start_time', np.nan)
        pro_match_dict['radiant_win'] = result.get('radiant_win', np.nan)

        # Populating players data 
        for player in result.get('players', {}):
            if not player:
                print(f'player data not available for match_id {match_id}')
            else:
                slot = player.get('player_slot', None)
                if slot is not None:  # To ensure only valid slots are updated
                    pro_match_dict[f"{slot}_account_id"] = player.get('account_id', np.nan)
                    pro_match_dict[f"{slot}_hero_id"] = player.get('hero_id', np.nan)
                else: 
                    print(f"slot is None for {result['match_id']}")

            
    list_pro_matches.append(pro_match_dict)
    
            
len(list_pro_matches)

12220

Remark: Processing of ten's of thousands of records is very fast

In [264]:
df_pro_matches = pd.DataFrame(list_pro_matches)
df_pro_matches

,match_id,radiant_team_id,radiant_name,dire_team_id,dire_name,duration,start_time,radiant_win,0_account_id,0_hero_id,1_account_id,1_hero_id,2_account_id,2_hero_id,3_account_id,3_hero_id,4_account_id,4_hero_id,128_account_id,128_hero_id,129_account_id,129_hero_id,130_account_id,130_hero_id,131_account_id,131_hero_id,132_account_id,132_hero_id
0,7.353447e+09,8765917.0,Shinigami Gaming,9175367.0,Business Club,1591.0,1.695750e+09,False,1.005381e+09,50.0,1.595129e+09,70.0,1.227193e+09,81.0,8.378903e+08,20.0,1.083774e+09,23.0,4.123204e+08,138.0,1.555070e+09,58.0,1.562326e+09,14.0,1.844555e+08,74.0,1.291450e+09,2.0
1,7.353388e+09,9175367.0,Business Club,8765917.0,Shinigami Gaming,1262.0,1.695747e+09,True,4.123204e+08,3.0,1.562326e+09,42.0,1.291450e+09,29.0,1.555070e+09,20.0,1.844555e+08,7.0,1.227193e+09,55.0,1.005381e+09,45.0,1.595129e+09,44.0,8.378903e+08,69.0,1.083774e+09,120.0
2,7.353377e+09,7300277.0,KZ TEAM,1061269.0,Vivo Keyd Stars,3103.0,1.695747e+09,False,1.710979e+08,114.0,1.206139e+08,46.0,2.023467e+08,110.0,9.294909e+07,53.0,2.717890e+07,104.0,8.682208e+07,95.0,1.313036e+08,107.0,8.485383e+07,119.0,8.130640e+07,37.0,1.075799e+08,137.0
3,7.353306e+09,9081007.0,Sibe Team,9180366.0,4Zoomers,1055.0,1.695744e+09,True,2.300986e+08,12.0,1.522852e+08,50.0,2.060974e+08,101.0,2.023922e+08,137.0,1.228175e+08,20.0,4.163729e+07,47.0,4.017926e+08,114.0,1.104487e+08,74.0,2.795593e+08,7.0,1.105834e+08,138.0
4,7.353294e+09,9087006.0,Blue lock,9088071.0,EYE Gaming,1635.0,1.695743e+09,True,3.412816e+08,96.0,1.466910e+09,44.0,3.621869e+08,3.0,1.963616e+08,121.0,1.589140e+09,74.0,1.404925e+09,85.0,1.801253e+08,137.0,1.304333e+08,67.0,3.399417e+08,37.0,1.139230e+08,58.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12215,7.138235e+09,8629315.0,Wiser Warriors,2443375.0,Foxy gaming,1862.0,1.683187e+09,False,1.627241e+08,90.0,1.170923e+09,67.0,2.029843e+08,38.0,9.923274e+08,40.0,1.232094e+09,69.0,8.059992e+07,101.0,1.045494e+09,51.0,3.014772e+08,84.0,1.103356e+09,104.0,1.813393e+08,11.0
12216,7.138227e+09,8936507.0,FenixTeam,8893825.0,Lucky Bulldogs,2025.0,1.683186e+09,True,1.533146e+09,73.0,1.479790e+09,90.0,2.883280e+08,99.0,1.479403e+09,87.0,1.300456e+09,22.0,1.459124e+09,10.0,1.423366e+09,28.0,1.459205e+09,106.0,1.459032e+09,5.0,1.458610e+09,21.0
12217,7.138207e+09,8980714.0,Team Tough,8944573.0,Hashtag.Reaper,1681.0,1.683185e+09,False,8.612229e+08,40.0,4.258834e+08,42.0,2.428356e+08,32.0,8.786137e+08,106.0,2.853195e+08,58.0,1.012409e+08,86.0,1.047812e+09,69.0,1.480632e+08,107.0,3.617944e+08,5.0,1.013255e+08,93.0
12218,7.138199e+09,8629318.0,Mad Monkeys,8629005.0,Flawless Goblins,1691.0,1.683184e+09,True,1.252616e+09,28.0,1.247011e+09,46.0,5.698168e+07,54.0,1.049956e+09,63.0,1.190299e+09,5.0,9.284998e+08,7.0,1.227183e+09,75.0,1.200964e+09,45.0,1.228356e+09,69.0,1.185918e+09,109.0


In [261]:
df_pro_matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12220 entries, 0 to 12219
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   match_id         12100 non-null  float64
 1   radiant_team_id  11986 non-null  float64
 2   radiant_name     11986 non-null  object 
 3   dire_team_id     11970 non-null  float64
 4   dire_name        11970 non-null  object 
 5   duration         12100 non-null  float64
 6   start_time       12100 non-null  float64
 7   radiant_win      12100 non-null  object 
 8   0_account_id     12100 non-null  float64
 9   0_hero_id        12100 non-null  float64
 10  1_account_id     12100 non-null  float64
 11  1_hero_id        12100 non-null  float64
 12  2_account_id     12100 non-null  float64
 13  2_hero_id        12100 non-null  float64
 14  3_account_id     12100 non-null  float64
 15  3_hero_id        12100 non-null  float64
 16  4_account_id     12100 non-null  float64
 17  4_hero_id   

In [262]:
# Storing to database 

import psycopg2
from psycopg2 import Error
from sqlalchemy.engine import create_engine, URL

url_object = URL.create(
    "postgresql+psycopg2",
    username='liuhaochen',
    host='localhost',
    port='5432',
    database='test'
)

engine = create_engine(url_object)

df_pro_matches.to_sql('pro_matches', engine, index=False, if_exists='append')

220

In [263]:
sql_df = pd.read_sql(
    "SELECT * FROM pro_matches",
    con=engine
)

sql_df

,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id,0_account_id,1_account_id,2_account_id,3_account_id,4_account_id,128_account_id,129_account_id,130_account_id,131_account_id,132_account_id,radiant_name,radiant_team_id,dire_name,dire_team_id,start_time,duration,radiant_win,match_id
0,4.0,110.0,23.0,77.0,62.0,136.0,9.0,128.0,42.0,13.0,1.185918e+09,1.227183e+09,1.200964e+09,1.228356e+09,9.284998e+08,1.005626e+09,9.486598e+08,1.144772e+09,9.130819e+08,2.414674e+08,Flawless Goblins,8629005.0,Dragon Blood,8629014.0,1.681448e+09,1799.0,False,7.106146e+09
1,126.0,6.0,128.0,9.0,71.0,100.0,31.0,51.0,46.0,123.0,1.526387e+09,1.427100e+09,1.517401e+09,1.513041e+09,1.529282e+09,1.525638e+09,1.524969e+09,1.524956e+09,1.525058e+09,1.525267e+09,Parallel eSports,8893837.0,Crew X,9018528.0,1.681447e+09,1893.0,True,7.106140e+09
2,62.0,29.0,69.0,95.0,13.0,1.0,87.0,77.0,128.0,21.0,1.144772e+09,9.486598e+08,1.005626e+09,9.130819e+08,2.414674e+08,1.185918e+09,1.227183e+09,1.228356e+09,1.200964e+09,9.284998e+08,Dragon Blood,8629014.0,Flawless Goblins,8629005.0,1.681445e+09,1657.0,False,7.106117e+09
3,9.0,44.0,129.0,64.0,101.0,19.0,84.0,21.0,46.0,71.0,1.525638e+09,1.525058e+09,1.524956e+09,1.524969e+09,1.525267e+09,1.526387e+09,1.513041e+09,1.517401e+09,1.427100e+09,1.529282e+09,Crew X,9018528.0,Parallel eSports,8893837.0,1.681445e+09,1634.0,False,7.106114e+09
4,87.0,69.0,49.0,11.0,33.0,10.0,39.0,96.0,86.0,83.0,8.130640e+07,8.485383e+07,8.593738e+07,8.682208e+07,1.179568e+08,2.024643e+08,3.999206e+08,1.484803e+08,3.923851e+08,3.025684e+08,Vivo Keyd Stars,1061269.0,Mad Kings Esports,8740097.0,1.681434e+09,1876.0,True,7.106006e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58597,90.0,67.0,38.0,40.0,69.0,101.0,51.0,84.0,104.0,11.0,1.627241e+08,1.170923e+09,2.029843e+08,9.923274e+08,1.232094e+09,8.059992e+07,1.045494e+09,3.014772e+08,1.103356e+09,1.813393e+08,Wiser Warriors,8629315.0,Foxy gaming,2443375.0,1.683187e+09,1862.0,False,7.138235e+09
58598,73.0,90.0,99.0,87.0,22.0,10.0,28.0,106.0,5.0,21.0,1.533146e+09,1.479790e+09,2.883280e+08,1.479403e+09,1.300456e+09,1.459124e+09,1.423366e+09,1.459205e+09,1.459032e+09,1.458610e+09,FenixTeam,8936507.0,Lucky Bulldogs,8893825.0,1.683186e+09,2025.0,True,7.138227e+09
58599,40.0,42.0,32.0,106.0,58.0,86.0,69.0,107.0,5.0,93.0,8.612229e+08,4.258834e+08,2.428356e+08,8.786137e+08,2.853195e+08,1.012409e+08,1.047812e+09,1.480632e+08,3.617944e+08,1.013255e+08,Team Tough,8980714.0,Hashtag.Reaper,8944573.0,1.683185e+09,1681.0,False,7.138207e+09
58600,28.0,46.0,54.0,63.0,5.0,7.0,75.0,45.0,69.0,109.0,1.252616e+09,1.247011e+09,5.698168e+07,1.049956e+09,1.190299e+09,9.284998e+08,1.227183e+09,1.200964e+09,1.228356e+09,1.185918e+09,Mad Monkeys,8629318.0,Flawless Goblins,8629005.0,1.683184e+09,1691.0,True,7.138199e+09


# Exploring Live League Games Endpoint

In [32]:
# Function to retrieve Json from Steam WebAPI
session = requests.Session()
session.params.update({'key': API_KEY})

@retry(tries=3, delay=2)
def fetch_live_league_games():
    try:
        url = f'{STEAM_URL}{LIVE_LEAGUE_GAMES}'
        res = session.get(url)
        match_details = res.json()
        if not match_details:
            raise ValueError("Empty dictionary, retrying...")
        else:
            return match_details
    except Exception as err:
        print("Did not get a response, retrying...")
        raise
    
game_data = fetch_live_league_games()
games = game_data['result']['games']

In [33]:
import numpy as np

live_league_games = []

def live_match_template():
    template = {
        'match_id': np.nan,
        'radiant_team_id': np.nan,
        'radiant_name': np.nan,
        'dire_team_id': np.nan,
        'dire_name': np.nan,
        'game_duration': np.nan,
        'start_time': np.nan
    }
    
    for i in list(range(0,5)) + list(range(128,133)):
        template[f"{i}_account_id"] = np.nan
        template[f"{i}_hero_id"] = np.nan
        
    return template




In [34]:


for row in games:
    
    match_dict = live_match_template()
    
    # Populate common fields
    match_dict['match_id'] = row.get('match_id', np.nan)
    match_dict['radiant_team_id'] = row.get('radiant_team', {}).get('team_id', np.nan)
    match_dict['radiant_name'] = row.get('radiant_team', {}).get('team_name', np.nan)
    match_dict['dire_team_id'] = row.get('dire_team', {}).get('team_id', np.nan)
    match_dict['dire_name'] = row.get('dire_team', {}).get('team_name', np.nan)
    match_dict['game_duration'] = row.get('scoreboard', {}).get('duration', np.nan)
    
    # Populate player data
    for team in ['radiant', 'dire']:
        for player in row.get('scoreboard', {}).get(team, {}).get('players', []):
            slot = player.get('player_slot', None)
            if slot is not None:  # To ensure only valid slots are updated
                match_dict[f"{slot}_account_id"] = player.get('account_id', np.nan)
                match_dict[f"{slot}_hero_id"] = player.get('hero_id', np.nan)
    
    live_league_games.append(match_dict)
    
    
live_league_games

[{'match_id': 7930240238,
  'radiant_team_id': nan,
  'radiant_name': nan,
  'dire_team_id': nan,
  'dire_name': nan,
  'game_duration': 0,
  'start_time': nan,
  '0_account_id': 1808007178,
  '0_hero_id': 114,
  '1_account_id': 1163929608,
  '1_hero_id': 44,
  '2_account_id': 1168169718,
  '2_hero_id': 63,
  '3_account_id': 1077414223,
  '3_hero_id': 64,
  '4_account_id': 119004907,
  '4_hero_id': 14,
  '128_account_id': 1064110501,
  '128_hero_id': 21,
  '129_account_id': 109383968,
  '129_hero_id': 11,
  '130_account_id': 1258621357,
  '130_hero_id': 26,
  '131_account_id': 162496533,
  '131_hero_id': 96,
  '132_account_id': 242217070,
  '132_hero_id': 30},
 {'match_id': 7930248050,
  'radiant_team_id': nan,
  'radiant_name': nan,
  'dire_team_id': nan,
  'dire_name': nan,
  'game_duration': 0,
  'start_time': nan,
  '0_account_id': 1792731012,
  '0_hero_id': 0,
  '1_account_id': 1779438847,
  '1_hero_id': 0,
  '2_account_id': 358387824,
  '2_hero_id': 0,
  '3_account_id': 206340530

In [35]:
live_df = pd.DataFrame(live_league_games)
live_df

,match_id,radiant_team_id,radiant_name,dire_team_id,dire_name,game_duration,start_time,0_account_id,0_hero_id,1_account_id,1_hero_id,2_account_id,2_hero_id,3_account_id,3_hero_id,4_account_id,4_hero_id,128_account_id,128_hero_id,129_account_id,129_hero_id,130_account_id,130_hero_id,131_account_id,131_hero_id,132_account_id,132_hero_id
0,7930240238,NaN,NaN,NaN,NaN,0.0,NaN,1.808007e+09,114.0,1.163930e+09,44.0,1.168170e+09,63.0,1.077414e+09,64.0,1.190049e+08,14.0,1.064111e+09,21.0,1.093840e+08,11.0,1.258621e+09,26.0,1.624965e+08,96.0,2.422171e+08,30.0
1,7930248050,NaN,NaN,NaN,NaN,0.0,NaN,1.792731e+09,0.0,1.779439e+09,0.0,3.583878e+08,0.0,2.063405e+08,0.0,1.236350e+09,0.0,1.223097e+08,0.0,2.442826e+08,0.0,1.115878e+09,0.0,1.032797e+09,0.0,1.429397e+09,0.0
2,7930243125,8629005.0,Flawless Goblins,8629324.0,Swift Knights,0.0,NaN,1.228356e+09,0.0,9.284998e+08,0.0,1.200964e+09,0.0,1.227183e+09,0.0,1.185918e+09,0.0,7.608292e+07,0.0,1.194936e+09,0.0,1.173255e+09,0.0,1.133046e+09,0.0,2.303080e+08,0.0
3,7930248954,9481763.0,Dominion,9487084.0,EYE Gaming,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [150]:
json_string = json.dumps(live_league_games)
with open('live_league_games.json','w') as file:
    file.write(json_string)

#### Varying Schemas for Live Games

1. matches containing teams without officially registered team_id will not contain the columns:
   1. 'radiant_team', 'dire_team'
2. matches which has not started will not contain the column 
   1. 'scoreboard'

## Exploring Match history Endpoint

Querying by match history allow's more customized querying option. However, output is limited to only the latest 500 matches.

In [18]:
PARAMS = {
    'game_mode':'2',
    'skill':'3',
    'date_min':'2023-03-12 00:00:00',
    'date_max':'2023-03-12 23:59:59',
    'matches_requested':'10',
}

## Possible to query tournament games by providing league_id

In [19]:
match_history = requests.get(f'{STEAM_URL}IDOTA2MATCH_570/GetMatchHistory/v1?'+'game_mode={game_mode}&skill={skill}&date_min={date_min}&date_max={date_max}&min_players=10&matches_requested={matches_num}&key={api_key}'.format(game_mode=PARAMS['game_mode'],
                                                                              skill=PARAMS['skill'], date_min=PARAMS['date_min'],
                                                                              date_max=PARAMS['date_max'], matches_num=PARAMS['matches_requested'],
                                                                              api_key=API_KEY))
match_history = match_history.json()

In [20]:
match_history['result']

{'status': 1,
 'num_results': 0,
 'total_results': 0,
 'results_remaining': 0,
 'matches': []}

In [21]:
match_history['result']['matches']

[]